In [44]:
import torch 
torch.manual_seed(0)

### True Data Generation Process 

In [46]:
n = 6
p = 5

true_mean = torch.zeros(p)  # mean vector of zeros
true_cov = torch.eye(p)     # identity covariance matrix
print(f"true_cov: {true_cov}")
X_data = torch.distributions.multivariate_normal.MultivariateNormal(
    loc=true_mean, # mean vector of zeros
    covariance_matrix=true_cov  # identity covariance matrix
).sample((n,))  # n x p
print(X_data.shape)  # should print torch.Size([100, 5])
print(true_cov.shape) # p x p 

# y = f(x) + noise(in this expeiment nois=0) and f(x) = sum of features of x
sigma_square_of_normal_noise = torch.tensor(0)  #  by sigma_square_of_normal_noise I mean the variance of the normal noise added to the outputs (y= f(x) + N(0, sigma^2))
y_data = X_data.sum(dim=1, keepdim=True)
print(y_data.shape)
print(X_data)

true_cov: tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]])
torch.Size([6, 5])
torch.Size([5, 5])
torch.Size([6, 1])
tensor([[ 0.2098,  0.0299,  1.7092, -0.7258, -1.4532],
        [-0.0476,  1.4998,  0.5431,  0.7393,  1.2532],
        [-0.4445,  0.8185,  0.0125,  0.9757, -0.4898],
        [-1.1727, -0.6870, -2.3349,  0.0940, -0.2021],
        [ 3.1257,  0.1784, -0.3368,  0.3260,  0.5352],
        [ 1.9733, -0.2075, -0.0306,  0.2597,  0.6116]])


In [47]:
# now my goal is to maximize the marginal log likelihood, so I will define my loss = - MLL (because we care to maximize MLL which is equivalent to minimizing - MLL)


#NOTE maybe try to see which values of sigmas give me N(0,I) since this is the data distribution I have assumed for y_data, and I will try to maximize p(y|X, sigmas) by changing sigmas

#NOTE: Is doing maximizing p(y|X, sigmas) equivalent to minimizing the difference between the predicted kernel matrix and I ?  (since y ~ N(0,I) i.e. like we were doing previously)


# intialization of the kernel matrix 

sigmas_pred = torch.randn(p,1, requires_grad=True)
learning_rate = 0.001

epochs = 10000
for epoch in range(epochs): 
    linear_kernel_pred = torch.ones(n,n) # these we do not care about their value they are just placeholder for now #TODO: investigate if the gradients will track chnages from one to k_i_j or not ( I think no )
    # instead of this loop;;;; I will do matrix multiplication 
    for i in range(n):
        for j in range(n): 
            k_i_j = 0
            for k in range(p):
                k_i_j += sigmas_pred[k]**2 * X_data[i,k] * X_data[j,k]
            linear_kernel_pred[i,j] = k_i_j
    print(f"linear_kernel_pred is {linear_kernel_pred}")
    # Computation of the loss 
    K = linear_kernel_pred + sigma_square_of_normal_noise * torch.eye(n) 
    K_inv = torch.linalg.inv(K)
    print(f"K_inv is {K_inv}")
    mu = torch.zeros(n,1) #TODO: how can we make it depend also on X_data
    sub = y_data - mu
    print(sub.shape)
    num = torch.exp(-0.5* (sub.T @ K_inv @ sub))
    print(f"num is {num}")
    print(f"det is {torch.det(K)}") #TODO: why is det 0? because K is singular? 
    den = torch.sqrt((2*torch.pi)**n * torch.det(K))
    print(f"den is {den}")
    MLL = num/den
    loss = - MLL
    loss.backward()

    # update sigmas_pred
    with torch.no_grad():
        sigmas_pred -= learning_rate * sigmas_pred.grad # p x 1 
        sigmas_pred.grad.zero_()

    if epoch % 10 == 0: 
        print(f"Probability became {MLL}")
        print(f"At epoch {epoch}, Covariance Matrix Predicted is : {K}")
    break


linear_kernel_pred is tensor([[ 4.5589, -3.3048,  1.2334, -0.2692, -1.2886, -1.5182],
        [-3.3048,  4.4320, -0.3171, -1.2863,  1.3525,  1.2298],
        [ 1.2334, -0.3171,  1.0569,  0.0880, -1.0001, -1.0358],
        [-0.2692, -1.2863,  0.0880,  1.9306, -1.8040, -1.1825],
        [-1.2886,  1.3525, -1.0001, -1.8040,  5.0300,  3.4078],
        [-1.5182,  1.2298, -1.0358, -1.1825,  3.4078,  2.5071]],
       grad_fn=<CopySlices>)
K_inv is tensor([[-1.2603e+08, -1.0131e+08,  1.8228e+07, -1.2528e+08,  8.3619e+07,
         -1.9184e+08],
        [-1.0131e+08, -8.1449e+07,  1.4654e+07, -1.0071e+08,  6.7223e+07,
         -1.5422e+08],
        [ 1.8227e+07,  1.4653e+07, -2.6363e+06,  1.8118e+07, -1.2094e+07,
          2.7745e+07],
        [-1.2528e+08, -1.0071e+08,  1.8119e+07, -1.2453e+08,  8.3121e+07,
         -1.9069e+08],
        [ 8.3620e+07,  6.7224e+07, -1.2095e+07,  8.3123e+07, -5.5483e+07,
          1.2729e+08],
        [-1.9184e+08, -1.5422e+08,  2.7747e+07, -1.9070e+08,  1.2729e+